In this notebook, we will show how to conduct energy simulation for an air-cooled data center with 5 data halls and a chiller plant with 5 chillers. In this case, the control variables is the chilled water supply temperature.

In [1]:
import numpy as np
from dctwin.interfaces.gym_envs import EPlusEnv
from dctwin.utils import config as env_config
from google.protobuf import json_format

from dctwin.utils import setup_logging, read_engine_config

/Users/ruihangwang/miniconda3/envs/dctwin/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1) Setup environment variables

In [2]:
engine_config = "example.prototxt"
env_config.eplus.engine_config_file = engine_config

(2) Read configuration files

In [3]:
config = read_engine_config(engine_config=engine_config)
setup_logging(config.logging_config, engine_config=engine_config)

2023-08-03 00:10:44.613 | INFO     | dctwin.utils.config:setup_logging:178 - Logging to /Users/ruihangwang/Desktop/Project/dctwin/tutorials/eplus/log/2023-08-03-00-10-44_eplus ...


(3) Build environment

In [4]:
env_config_name = config.WhichOneof("EnvConfig")
env_params = json_format.MessageToDict(
    getattr(config, env_config_name).env_params,
    preserving_proto_field_name=True,
)
env = EPlusEnv(
    config=getattr(config, env_config_name),
    reward_fn=None,
    schedule_fn=None,
    **env_params,
)

2023-08-03 00:10:47.230 | INFO     | dctwin.interfaces.gym_envs.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc1_supply_temperature
2023-08-03 00:10:47.231 | INFO     | dctwin.interfaces.gym_envs.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc2_supply_temperature
2023-08-03 00:10:47.231 | INFO     | dctwin.interfaces.gym_envs.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc3_supply_temperature
2023-08-03 00:10:47.232 | INFO     | dctwin.interfaces.gym_envs.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc4_supply_temperature
2023-08-03 00:10:47.232 | INFO     | dctwin.interfaces.gym_envs.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc5_supply_temperature
2023-08-03 00:10:47.235 | INFO     | dctwin.interfaces.gym_envs.base_env:_set_simulation_time:106 - Using pre-set simulation time
2023-08-03 00:10:47.304 | INFO     | dctwin.backends.eplus.core:_set_up_socket:79 - socket listening on host.docker.internal:53492


(4) Run EnergyPlus simulation

In [5]:
water_supply_sp = 12.0
env.reset()
done = False
while not done:
    act = np.array([water_supply_sp])
    obs, rew, done, truncated, info = env.step(act)

2023-08-03 00:10:49.478 | INFO     | dctwin.backends.eplus.core:_parse_idf_and_gen_bcvtb_config:115 - Parsing IDF file...
2023-08-03 00:10:50.540 | INFO     | dctwin.backends.eplus.core:_parse_idf_and_gen_bcvtb_config:134 - Generating BCVTB Config ...
2023-08-03 00:10:50.543 | INFO     | dctwin.models.eplus.eplus:save_cfg_xml:760 - no internal schedule config added
2023-08-03 00:10:50.596 | INFO     | dctwin.backends.core:run_container:68 - docker mount: /Users/ruihangwang/Desktop/Project/dctwin/tutorials/eplus/log/2023-08-03-00-10-44_eplus/eplus_output/episode-1
2023-08-03 00:10:50.597 | INFO     | dctwin.backends.core:run_container:69 - docker run: /bin/bash -c /usr/local/EnergyPlus-9-5-0/energyplus -w SGP_Singapore.486980_IWEC.epw -r 5DC_4CH.idf
2023-08-03 00:10:56.575 | INFO     | dctwin.backends.eplus.core:_run_backend:203 - Got connection from ('127.0.0.1', 53495)
2023-08-03 00:11:14.791 | DEBUG    | dctwin.backends.eplus.core:receive_status:268 - Came to the end of one episode, 